<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/Decoder_only_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch -q

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

set_seed(42)

model_id='distilbert/distilgpt2'

tokenizer=AutoTokenizer.from_pretrained(model_id)
model=AutoModelForCausalLM.from_pretrained(model_id)

tokenizer.pad_token=tokenizer.eos_token
model.config.pad_token_id=tokenizer.eos_token_id

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [3]:
prompt='The capital of france is'

inputs=tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
  outputs=model(**inputs)

logits=outputs.logits
next_token_logits=logits[0, -1, :]
probs=torch.softmax(next_token_logits, dim=-1)

top_k=10
top_probs, top_ids=torch.topk(probs, top_k)

for prob,token_id in zip(top_probs, top_ids):
  token=tokenizer.decode(token_id)
  print(f'{token}: {prob.item():.4f}')

 the: 0.1477
 a: 0.0732
 in: 0.0306
 now: 0.0191
 located: 0.0149
 an: 0.0112
 not: 0.0106
 one: 0.0102
 Paris: 0.0100
 also: 0.0099


##Greedy Decoding

In [4]:
next_token_id=torch.argmax(probs).item()
next_token=tokenizer.decode(next_token_id)
print('Selected token:', repr(next_token))
print('New text',prompt+next_token)

Selected token: ' the'
New text The capital of france is the


In [5]:
def manual_greedy(prompt,steps=20):
  text=prompt

  for _ in range(steps):
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
      outputs=model(**inputs)
    logits=outputs.logits[0,-1,:]
    probs=torch.softmax(logits,dim=-1)

    next_token_id=torch.argmax(probs).unsqueeze(0)
    next_token=tokenizer.decode(next_token_id)
    text+=next_token
  return text


In [6]:
print(manual_greedy('Once upon a time',steps=30))

Once upon a time of war, the United States was the only country in the world to have a military presence. The United States was the only country in the world to


##Sampling

In [7]:
def manual_sampling(prompt,steps=20):
  text=prompt

  for _ in range(steps):
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
      outputs=model(**inputs)
    logits=outputs.logits[0,-1,:]
    probs=torch.softmax(logits,dim=-1)

    next_token_id=torch.multinomial(probs,num_samples=1)
    next_token=tokenizer.decode(next_token_id)

    text+=next_token
  return text

In [8]:
print(manual_sampling('Once upon a time',steps=30))

Once upon a time, you take that advantage as a preparation lesson to succeed. When you start playing hard enough, you will begin to learn all sorts of new skills and


##temperature,top-k,top-p

In [9]:
def manual_temperature_sampling(prompt,steps=20,temperature=0.7):
  text=prompt

  for _ in range(steps):
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
      outputs=model(**inputs)
    logits=outputs.logits[0,-1,:]
    scaled_logits=logits/temperature
    probs=torch.softmax(logits,dim=-1)

    next_token_id=torch.multinomial(probs,num_samples=1)
    next_token=tokenizer.decode(next_token_id)

    text+=next_token
  return text

In [10]:
for temp in [0.3,0.7,1.2]:
  print('Temperature',temp)
  print(manual_temperature_sampling('Once upon a time ',steps=30,temperature=temp))

Temperature 0.3
Once upon a time ????? what is most interesting is how significant it is *now* Endurance whilst still running!
The coaching effort is shown from various levels of the
Temperature 0.7
Once upon a time   something along the other side had died. We must sit at the dais. Nice in many respects. Great. I was right!
12
Temperature 1.2
Once upon a time 三��■■■■■■■■■■■■■■■■■■■■■■■■■■■


##top-k samplng

In [11]:
def manual_top_k_sampling(prompt,steps=20,top_k=50,temperature=0.7):
  text=prompt

  for _ in range(steps):
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
      outputs=model(**inputs)
    logits=outputs.logits[0,-1,:]
    logits=logits/temperature
    top_values,top_indices=torch.topk(logits,top_k)
    top_probs=torch.softmax(top_values,dim=-1)
    sampled_index=torch.multinomial(top_probs,num_samples=1)
    next_token_id=top_indices[sampled_index]
    next_token_id=torch.multinomial(probs,num_samples=1)
    next_token=tokenizer.decode(next_token_id)

    text+=next_token
  return text

In [12]:
for k in [0.3,0.7,1.2]:
  print('Temperature',temp)
  print(manual_top_k_sampling('Once upon a time ',steps=30,temperature=temp,top_k=50))

Temperature 1.2
Once upon a time  the Sark capital La E al Stras Great castle on France Column a Santiago the Cas Moscow territory to Athens surrounded especially raided a London decre the Paris Lisbon Paris
Temperature 1.2
Once upon a time  the 25 another Rome v in the in one within playing almost about Glen Cologne filled also tomorrow Also a laying el in Z not Vienna spear B ONE Sant
Temperature 1.2
Once upon a time  at R discovered now including often the T Rom there a the important the Jordan their Romania Barcelona a Christ H know the France a a the considered catapult outside


##top-p/nucleus sampling

In [26]:
def manual_top_p_sampling(prompt,steps=20,top_p=0.9,temperature=0.7):
  text=prompt

  for _ in range(steps):
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
      outputs=model(**inputs)
    logits=outputs.logits[0,-1,:]
    logits=logits/temperature
    probs=torch.softmax(logits,dim=-1)
    sorted_probs,sorted_indices=torch.sort(probs,descending=True)
    cumulative_probs=torch.cumsum(sorted_probs,dim=-1)
    mask=cumulative_probs<=top_p
    mask[0]=True

    filtered_probs=sorted_probs[mask]
    filtered_indices=sorted_indices[mask]

    filtered_probs=filtered_probs/filtered_probs.sum()
    sampled_index=torch.multinomial(filtered_probs,num_samples=1)
    next_token_id=filtered_indices[sampled_index]

    next_token=tokenizer.decode(next_token_id)
    text+=next_token
  return text

In [27]:
for p in [0.5,0.8,0.95]:
  print('Top-p',p)
  print(manual_top_p_sampling('Once upon a time ',steps=40,temperature=0.8,top_p=p))

Top-p 0.5
Once upon a time   I had a feeling that the old man was going to be a big part of the story.
I was very surprised by the way he was portrayed. I was not sure if he was going
Top-p 0.8
Once upon a time vernacular change, we can't be happy. But at the same time, it's still a good time to check your smartphone and use it.




The first thing you need
Top-p 0.95
Once upon a time   the moment I was in the car I noticed a white of the car.
As I sat down the door, I noticed it was white.   I immediately noticed that the lights in the car


##Hugging Face generate

In [28]:
def hf_generate(prompt,max_new_tokens=80,**kwargs):
  inputs=tokenizer(prompt,return_tensors='pt').to(device)
  outputs=model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      pad_token_id=tokenizer.eos_token_id,
      **kwargs
  )
  return tokenizer.decode(outputs[0],skip_special_tokens=True)

In [29]:
prompt='Artificial intelligence is'

In [30]:
output=hf_generate(
    prompt,
    do_sample=False,
)
print(repr(output))

'Artificial intelligence is a new technology that is being developed by researchers at the University of California, Berkeley.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'


In [31]:
output=hf_generate(
    prompt,
    do_sample=True,
)
print(repr(output))

'Artificial intelligence is one of the most powerful devices in the world and it has made headlines in the past year due to its remarkable capabilities and its enormous capabilities (see below). But there have been setbacks. Despite its capabilities, the company did not reach the heights necessary to make it to the next level.\n\n\nAccording to the new research conducted by ICTAI, it is not the first time a machine has'


In [32]:
output=hf_generate(
    prompt,
    do_sample=True,
    temperature=0.7,
)
print(repr(output))

'Artificial intelligence is a social service that is not intended to be a commodity or a currency. It is designed to serve people.\n\n\n\nIn a recent interview, Dr. David Schwerin, a neurosurgeon with the University of Texas at Austin, said that his research is "a powerful tool for the development of a social service that will be able to function in a variety of ways."\nIn'


In [33]:
output=hf_generate(
    prompt,
    do_sample=True,
    top_k=50,
    temperature=0.8,
)
print(repr(output))

"Artificial intelligence is a new technology that can be used to improve our lives and the futures of the world.\n\n\n\n\n\nThe AI system, the Future, is already being used in the world of artificial intelligence, but it has not yet been shown to be accurate, and this is a clear indication that it is being used in the future.\nThe future will be defined by the world's future,"
